# Week 2: Feature Engineering + ML Models

- **Training:** 2013-01-01 to 2013-12-31
- **Test:** 2014-01-01 to 2014-03-31

Feature engineering is the process of transforming raw data into inputs (features) that improve machine learning model performance. It includes creating, selecting, scaling, and encoding variables to better capture patterns in the data.

In [ ]:
# Import libraries for ML models, evaluation, and plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load & Prepare Data

In [ ]:
# Load base sales, oil prices, and holidays
df = pd.read_csv("data/feature_engineered_timeseries.csv", parse_dates=["date"])
oil = pd.read_csv("data/oil.csv", parse_dates=["date"]).rename(columns={"dcoilwtico": "oil_price"})
holidays = pd.read_csv("data/holidays.csv", parse_dates=["date"])

# Merge external data and create holiday flags
df = df.merge(oil, on="date", how="left")
holiday_dates = holidays["date"].unique()
df["is_holiday"] = df["date"].isin(holiday_dates).astype(int)
nat_dates = holidays[holidays["locale"]=="National"]["date"].unique()
df["is_national_holiday"] = df["date"].isin(nat_dates).astype(int)

print(f"Shape: {df.shape}, columns: {len(df.columns)}")

## 2. Feature Engineering

In [ ]:
# Create calendar, oil, sales lag/rolling features, and holiday proximity
df = df.sort_values("date").reset_index(drop=True)

df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["quarter"] = df["date"].dt.quarter
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

df["oil_lag_1"] = df["oil_price"].shift(1)
df["oil_rolling_mean_7"] = df["oil_price"].rolling(7, min_periods=1).mean()
df["oil_change"] = df["oil_price"].pct_change()

df["lag_14"] = df["unit_sales"].shift(14)
df["lag_21"] = df["unit_sales"].shift(21)
df["rolling_mean_7"] = df["unit_sales"].rolling(7, min_periods=1).mean()
df["rolling_mean_30"] = df["unit_sales"].rolling(30, min_periods=1).mean()

hol_ar = pd.Series(holiday_dates).sort_values()
df["days_to_next_holiday"] = df["date"].apply(
    lambda d: (hol_ar[hol_ar > d].iloc[0] - d).days
    if (hol_ar > d).any() else np.nan
)
df["weekend_holiday"] = df["is_weekend"] * df["is_holiday"]

print(f"Features: {len(df.columns)}")

## 3. Why These Features?

| Group | Columns | Why |
|---|---|---|
| **Calendar** | `day_of_month`, `week_of_year`, `quarter`, `is_month_start/end` | Monthly/quarterly patterns — paydays, end-of-month effects, seasonal retail shifts |
| **Oil** | `oil_lag_1`, `oil_rolling_mean_7`, `oil_change` | Yesterday's price, recent avg, daily change — oil impacts Ecuadorian economy and consumer spending |
| **Sales lags** | `lag_14`, `lag_21` | 2 and 3 weeks ago — weekly seasonality makes same-weekday lags most informative |
| **Sales rolling** | `rolling_mean_7`, `rolling_mean_30` | Recent trend level — smooths noise, captures short-term direction |
| **Holiday proximity** | `days_to_next_holiday` | Sales often ramp up before events (e.g., Christmas shopping) |
| **Interaction** | `weekend_holiday` | `is_weekend * is_holiday` — weekends that are also holidays behave differently |

## 4. Train/Test Split

In [ ]:
# Split into train/test and drop rows with NaN from lag features
target = "unit_sales"
features = [c for c in df.columns if c not in ("date", target)]

train = df[df["date"] < "2014-01-01"]
test = df[(df["date"] >= "2014-01-01") & (df["date"] <= "2014-03-31")]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

train_mask = X_train.notna().all(axis=1)
test_mask = X_test.notna().all(axis=1)
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_test, y_test = X_test[test_mask], y_test[test_mask]
test_dates = test.loc[test_mask, "date"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Visualise target distribution
plt.hist(y_train, bins=30, edgecolor="black", alpha=0.7)
plt.title("Distribution of Unit Sales (Train)")
plt.show()

## 5. Evaluation Helpers

In [ ]:
# Utility functions for scoring and plotting forecasts
def evaluate(true, pred, name):
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true, pred)
    r2 = r2_score(true, pred)
    print(f"{name:20s}  MSE: {mse:.1f}  RMSE: {rmse:.1f}  MAE: {mae:.1f}  R²: {r2:.4f}")
    return {"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

results = []
preds = {}

def plot_forecast(dates, true, pred, title):
    plt.plot(dates, true, label="Actual", color="orange")
    plt.plot(dates, pred, label="Forecast", color="red", linestyle="--")
    plt.title(title)
    plt.legend()
    plt.show()

def plot_importance(importances, title):
    top = importances.head(10)
    plt.barh(range(len(top)), top.values, color="#3498db", edgecolor="black")
    plt.yticks(range(len(top)), top.index)
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 6. Model 1: Linear Regression

In [ ]:
# Linear baseline model
m = LinearRegression().fit(X_train, y_train)
pred = np.clip(m.predict(X_test), 0, None)
preds["Linear Regression"] = pred
plot_forecast(test_dates, y_test, pred, "Linear Regression")
results.append(evaluate(y_test, pred, "Linear Regression"))

## 7. Model 2: Random Forest

In [ ]:
# Random Forest with 300 trees
m = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
m.fit(X_train, y_train)
pred = np.clip(m.predict(X_test), 0, None)
preds["Random Forest"] = pred
plot_forecast(test_dates, y_test, pred, "Random Forest")
results.append(evaluate(y_test, pred, "Random Forest"))

imps = pd.Series(m.feature_importances_, index=features).sort_values(ascending=False)
plot_importance(imps, "Random Forest Feature Importance")

## 8. Model 3: XGBoost

In [ ]:
# XGBoost with sequential boosting for non-linear patterns
m = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                 random_state=42, n_jobs=-1, objective="reg:squarederror")
m.fit(X_train, y_train)
pred = np.clip(m.predict(X_test), 0, None)
preds["XGBoost"] = pred
plot_forecast(test_dates, y_test, pred, "XGBoost")
results.append(evaluate(y_test, pred, "XGBoost"))

imps = pd.Series(m.feature_importances_, index=features).sort_values(ascending=False)
plot_importance(imps, "XGBoost Feature Importance")

## 9. Model Comparison

In [ ]:
# Rank models by RMSE and overlay all forecasts
rdf = pd.DataFrame(results).sort_values("RMSE")
print(rdf.to_string(index=False))

plt.bar(rdf["Model"], rdf["RMSE"], color="#3498db", edgecolor="black")
plt.title("ML Model Comparison (RMSE)")
plt.ylabel("RMSE")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.plot(test_dates, y_test, label="Actual", color="black", linewidth=2)
for r in results:
    plt.plot(test_dates, preds[r["Model"]], label=r["Model"], linestyle="--", alpha=0.7)
plt.title("Forecast Comparison: All Models")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Summary

Expected XGBoost > RF > LR, 
Actual is LR > XGBoost > RF. 

The notebooks were never executed — the expected ranking was a hypothesis that didn't hold empirically.

Likely due to small dataset (334 samples) making simpler models win.